# <a id='toc1_'></a>[C50 / brain fm - date of diagnosis vs follow up / treatment dates](#toc0_)

**❓ Fragestellung**
> Die meisten Hirnmetastasen (als Folgeereignis_Fernmetastase), die man für die Kalenderjahre 2020-2023 erwarten würde, entstehen bei Frauen, die ihre Primärdiagnose weit vor 2020 hatten. Beziehen sich die klinischen ZfKD Daten, einschließlich Therapien und Folgeereignissen, tatsächlich aber nur auf Patienten, die auch während 2020-2023 einen Primärtumor hatten (z_tum_id)? Dieser Filter würde einen großen Teil der Events in „Folgeereignis_Fernmetastase“ entfernen, so dass die Zahlen in den deutschlandweiten Registerdaten deutlich unterhalb den Daten aus der Versorgung liegen. Wir wüssten dann z.B. nicht, wie viele Hirnmetastasen pro Kalenderjahr versorgt werden müssen.

**⚖️ Analyse**
- Filter: alle Tumore mit `C50` und einer zugeordneten FM mit Lokalisation `BRA` **(2.1k)**
- _follow up_
  - gezählt sind alle Folgeereignisse zu den Tumoren im Filter **(9k)**
  - die gezeigten 5 Kategorien/Gruppen sind immer disjunkt
  - bei 96% liegen Diagnose und Folgeereignis > 2020
  - Altfälle (Gruppe 1) gibt in einigen GTDS Registern
  - Folgeereignisse vor 2020 (Gruppe 3) kommen fast nur aus NW
- _treatments_
  - gezählt sind alle Tumoren im Filter **(2.1k)**
  - bei 96% liegen Diagnose und erste Therapie >2020
  - Gruppe 5 sind Fälle ohne Therapieangabe (3%)
  - Altfälle sind selten, Therapien <2020 bei Fällen nach 2020 kommen gar nicht vor

**Table of contents**<a id='toc0_'></a>    
- [C71 brain - date of diagnosis vs follow up / treatment dates](#toc1_)    
  - [📆 data as of](#toc1_1_)    
  - [analysis](#toc1_2_)    
    - [follow up](#toc1_2_1_)    
    - [treatment](#toc1_2_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import os
from pathlib import Path
import pandas as pd
import duckdb as ddb
from connection_helper import sql
import numpy as np
from pandas_plots import tbl, pls, hlp
import datetime as dt

hlp.show_package_version(["pygwalker"])
os.environ['THEME']='light'
os.environ['DEBUG']='0'

dir_db=Path("C://temp") if hlp.get_os(hlp.OperatingSystem.WINDOWS) else Path(os.path.expanduser("~/tmp"))

file_db_clin = dir_db/'workflow/2025-10-30_data_clin.duckdb'

if not file_db_clin.exists():
    raise(FileNotFoundError(f"File {file_db_clin} not found"))

🐍 3.12.8 | 📦 pygwalker: 0.4.9.15 | 📦 pandas: 2.3.3 | 📦 numpy: 1.26.4 | 📦 duckdb: 1.4.1 | 📦 pandas-plots: 0.20.4 | 📦 connection-helper: 0.13.1


In [2]:
con = ddb.connect(file_db_clin, read_only=True)
_=con.execute("PRAGMA disable_progress_bar;")

## <a id='toc1_1_'></a>[📆 data as of](#toc0_)

In [3]:
sql.print_meta(file_db_clin)

sqlite db file:          2025-06-24_data_clin.duckdb
data tag:                v2.3
last kkr data import:    2025-05-27
sql table created:       2025-09-25 11:30:28
doi:                     10.18444/5.03.01.0005.0021.0001
document created:        2025-10-08 11:47:05


## <a id='toc1_2_'></a>[analysis](#toc0_)

In [4]:
t_filter = con.sql("""--sql
    select tum.z_tum_id, z_kkr_label, z_icd10, Diagnosedatum, z_first_treatment_after_days, z_first_treatment,
    from Tumor tum
    join Folgeereignis_Fernmetastase fm on tum.z_tum_id = fm.z_tum_id
    where z_icd10_3d = 'C50'
    and Lokalisation = 'BRA'
    """)

n=t_filter.to_df().shape[0]
print(f"Anzahel Tumore im Filter: n = {n:_}")
tbl.descr_db(t_filter, "filter")

Anzahel Tumore im Filter: n = 2_119
🗄️ filter	2_119, 6
	("z_tum_id, z_kkr_label, z_icd10, Diagnosedatum, z_first_treatment_after_days, z_first_treatment")
┌──────────────────────────────────────┬─────────────┬─────────┬───────────────┬──────────────────────────────┬───────────────────┐
│               z_tum_id               │ z_kkr_label │ z_icd10 │ Diagnosedatum │ z_first_treatment_after_days │ z_first_treatment │
│               varchar                │   varchar   │ varchar │     date      │            int32             │      varchar      │
├──────────────────────────────────────┼─────────────┼─────────┼───────────────┼──────────────────────────────┼───────────────────┤
│ ab339db0-6010-46cb-b46f-8ce37969b687 │ 05-NW       │ C50.4   │ 2020-07-15    │                           67 │ sy                │
│ 1941c0ff-58db-4c92-ab10-f06bd9779dd6 │ 11-BE       │ C50.4   │ 2023-01-15    │                          205 │ op                │
│ c6b6775a-ff7c-4836-a13c-f393455fe7d1 │ 05-NW       

### <a id='toc1_2_1_'></a>[follow up](#toc0_)

In [5]:
db_diag_fo =con.sql("""--sql
    select
            --tum.z_tum_id,
            --z_icd10, Diagnosedatum,
            --Datum_Folgeereignis,
            z_kkr_label,
            case 
                when Diagnosedatum < '2020-01-01' and Datum_Folgeereignis < '2020-01-01' then '1_all<2020'
                when Diagnosedatum < '2020-01-01' and Datum_Folgeereignis >= '2020-01-01' then '2_diag<2020'
                when Diagnosedatum >= '2020-01-01' and Datum_Folgeereignis < '2020-01-01' then '3_fo<2020'
                when Diagnosedatum >= '2020-01-01' and Datum_Folgeereignis >= '2020-01-01' then '4_all>=2020'
                else '9_unknown'
            end as categ_diag_fo
    from t_filter tum
    join Folgeereignis fol on fol.z_tum_id = tum.z_tum_id
""")
n_fo = db_diag_fo.to_df().shape[0]
print(f"Anzahl Folgeereignisse im Filter: n = {n_fo:_}")
# tbl.descr_db(db_diag_fo, "diag_fo")

Anzahl Folgeereignisse im Filter: n = 9_483


In [6]:
pls.plot_stacked_bars(
    db_diag_fo.to_df(),
    relative=True,
    show_total=True,
    orientation="h",
    show_pct_bar=True,
    width=1500,
    height=600,
    caption="events",
    kkr_col="z_kkr_label",
)
tbl.pivot_df(db_diag_fo.to_df())

categ_diag_fo,1_all<2020,2_diag<2020,3_fo<2020,4_all>=2020,Total
z_kkr_label,,,,,
02-HH,0,0,0,12 (0.1%),12 (0.1%)
05-NW,0,0,152 (1.6%),2_122 (22.4%),2_274 (24.0%)
06-HE,32 (0.3%),66 (0.7%),0,338 (3.6%),436 (4.6%)
07-RP,0,0,0,448 (4.7%),448 (4.7%)
08-BW,0,0,0,3_227 (34.0%),3_227 (34.0%)
09-BY,0,2 (0.0%),7 (0.1%),1_058 (11.2%),1_067 (11.3%)
11-BE,0,0,0,220 (2.3%),220 (2.3%)
12-BB,0,0,0,241 (2.5%),241 (2.5%)
13-MV,11 (0.1%),8 (0.1%),0,292 (3.1%),311 (3.3%)


### <a id='toc1_2_2_'></a>[treatment](#toc0_)

In [7]:
db_diag_treat = con.sql("""--sql
    select
            z_kkr_label,
            case
                -- Calculation is now: Diagnosedatum + z_first_treatment_after_days
                when Diagnosedatum < '2020-01-01' and (Diagnosedatum + z_first_treatment_after_days) < '2020-01-01' then '1_all<2020'
                when Diagnosedatum < '2020-01-01' and (Diagnosedatum + z_first_treatment_after_days) >= '2020-01-01' then '2_diag<2020'
                when Diagnosedatum >= '2020-01-01' and (Diagnosedatum + z_first_treatment_after_days) < '2020-01-01' then '3_treat<2020'
                when Diagnosedatum >= '2020-01-01' and (Diagnosedatum + z_first_treatment_after_days) >= '2020-01-01' then '4_all>=2020'
                when z_first_treatment is null then '5_no_treat'
                else '9_unknown'
            end as categ_diag_treat,
            Diagnosedatum,
            z_first_treatment_after_days,
            z_tum_id,
    from t_filter tum
""")

# tbl.descr_db(db_diag_treat, "diag_treat")

In [8]:
_df = db_diag_treat.to_df().iloc[:,:2]

pls.plot_stacked_bars(
    _df,
    relative=True,
    show_total=True,
    orientation="h",
    show_pct_bar=True,
    width=1500,
    height=600,
    caption="tumors",
    kkr_col="z_kkr_label",
    )
tbl.pivot_df(_df)

categ_diag_treat,1_all<2020,2_diag<2020,4_all>=2020,5_no_treat,Total
z_kkr_label,,,,,
02-HH,0,0,8 (0.4%),0,8 (0.4%)
05-NW,0,0,426 (20.1%),12 (0.6%),438 (20.7%)
06-HE,12 (0.6%),3 (0.1%),94 (4.4%),7 (0.3%),116 (5.5%)
07-RP,0,0,112 (5.3%),2 (0.1%),114 (5.4%)
08-BW,0,0,515 (24.3%),7 (0.3%),522 (24.6%)
09-BY,0,0,314 (14.8%),16 (0.8%),330 (15.6%)
11-BE,0,0,100 (4.7%),3 (0.1%),103 (4.9%)
12-BB,0,0,87 (4.1%),0,87 (4.1%)
13-MV,2 (0.1%),0,67 (3.2%),2 (0.1%),71 (3.4%)


In [10]:
if os.getenv("DEBUG") == "1":
    tbl.descr_db(
        db_diag_treat
        .filter("left(categ_diag_treat,1) = '9'")
        ,"treatment"
    )
    hlp.get_tum_details("8f92095a-e3e7-45b2-b290-0a7cefdc5a20", con=con)